## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-11-10-54-00 +0000,wsj,European TTF Gas Rises as Storage Withdrawals ...,https://www.wsj.com/finance/commodities-future...
1,2026-11-11-10-52-58 +0000,nypost,Mom of black Ole Miss freshman speaks out as s...,https://nypost.com/2026/11/11/us-news/mom-of-o...
2,2026-11-11-10-49-00 +0000,wsj,McCormick’s $65 Billion Unilever Foods Tie-Up ...,https://www.wsj.com/business/deals/mccormicks-...
3,2026-11-11-10-45-25 +0000,nypost,‘One-in-a-million’ bright yellow fish discover...,https://nypost.com/2026/11/11/us-news/bright-y...
4,2026-11-11-10-41-00 +0000,wsj,Siemens Energy Shares Jump on Steeper Profit T...,https://www.wsj.com/business/earnings/siemens-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2661/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
127,trump,30
122,nvidia,26
487,huang,20
101,new,19
80,house,17


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
165,2026-11-11-00-07-28 +0000,nypost,Chinese ‘spy’ Alice Huang met Nvidia CEO Jense...,https://nypost.com/2026/11/10/business/alice-h...,156
449,2026-11-10-11-59-50 +0000,nypost,FBI is ‘looking into’ Nvidia boss Jensen Huang...,https://nypost.com/2026/11/10/us-news/fbi-is-l...,145
304,2026-11-10-19-21-16 +0000,nypost,Nvidia loses nearly $450B in two-day wipeout a...,https://nypost.com/2026/11/10/business/nvidia-...,141
275,2026-11-10-20-22-14 +0000,nypost,Jensen Huang’s ties to suspected Chinese spy h...,https://nypost.com/2026/11/10/business/jensen-...,135
239,2026-11-10-21-35-09 +0000,nypost,Nvidia CEO Jensen Huang’s ‘spy’ affair could s...,https://nypost.com/2026/11/10/business/nvidia-...,126


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
165,156,2026-11-11-00-07-28 +0000,nypost,Chinese ‘spy’ Alice Huang met Nvidia CEO Jense...,https://nypost.com/2026/11/10/business/alice-h...
127,107,2026-11-11-02-15-00 +0000,wsj,House Republicans elected Rep. Jim Jordan of O...,https://www.wsj.com/politics/jim-jordan-wins-s...
349,103,2026-11-10-17-36-51 +0000,nypost,Gas prices drop below $4 a gallon for first ti...,https://nypost.com/2026/11/10/business/gas-pri...
156,78,2026-11-11-00-29-45 +0000,latimes,Meghan Markle's father Thomas Markle Sr. cut o...,https://www.latimes.com/entertainment-arts/sto...
152,72,2026-11-11-00-34-00 +0000,wsj,Defense Secretary Pete Hegseth gave a Pentagon...,https://www.wsj.com/politics/national-security...
412,55,2026-11-10-14-20-09 +0000,cbc,Anthropic eyes $1 trillion US valuation in hot...,https://www.cbc.ca/news/business/anthropic-ipo...
353,52,2026-11-10-17-30-00 +0000,wapo,Trump-appointed Fed member signals big interes...,https://www.washingtonpost.com/business/2026/1...
1,51,2026-11-11-10-52-58 +0000,nypost,Mom of black Ole Miss freshman speaks out as s...,https://nypost.com/2026/11/11/us-news/mom-of-o...
277,50,2026-11-10-20-20-04 +0000,bbc,US Army soldier suspected of spying for Russia...,https://www.bbc.co.uk/news/articles/c6d7q77xgw...
212,49,2026-11-10-22-30-00 +0000,wsj,"Senate Majority Leader John Thune (R., S.D.) w...",https://www.wsj.com/politics/policy/john-thune...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
